# Laboratory: File Manipulation and Document Ingestion in NLP Pipelines

## Overview
In natural language processing (NLP) pipelines, raw corpora are rarely delivered in clean, in-memory string formats. Practical workflows require ingesting, cleaning, and transforming textual data across heterogeneous formats:
- Plain text (`.txt`) streams with explicit character encodings (e.g., UTF-8 for non-Latin and Arabic scripts).
- Structured tabular corpora (`.csv`, `.xlsx`) requiring feature extraction, header management, and batch serialization.
- Compiled binary document formats (`.pdf`) requiring structural parsing, text extraction, page manipulation, and stream concatenation.
- Lexical pattern matching and regex-based token filtering (`re.search`, `re.findall`).

This notebook demonstrates the standard paradigms for file I/O, cursor stream manipulation, character encoding safety, structured data engineering, and binary document extraction.

## 1. Low-Level File I/O: `open()`, `read()`, and `close()`

Traditional file input/output in Python interacts directly with the operating system file descriptor table. Opening a file in read mode (`'r'`) allocates a file handle and positions the read cursor at byte offset `0`.

When reading files using standard assignment:
1. `open(path, 'r')`: Allocates the file descriptor.
2. `f.read()`: Reads the entire payload into RAM as a string and advances the stream cursor to End-Of-File (EOF).
3. `f.close()`: Explicitly releases the operating system resource.

*Caution:* If an unhandled exception occurs prior to `f.close()`, the file descriptor remains allocated in system memory, creating resource leaks.

In [1]:
# Open the file in standard read-only text mode ('r')
# Note: open() returns a file descriptor / stream object
f = open("./data/data.txt", "r")

# Read the entire content of the file into memory as a single string
contenu = f.read()

# Output the retrieved file contents
print(contenu)

# Explicitly close the file stream to release the operating system resource descriptor
# Note: If an error occurs prior to close(), the descriptor remains locked in memory
f.close()

NLP.
Data analysis.
New line added.



## 2. Deterministic Resource Management with Context Managers (`with`)

The Python context manager (`with` statement) adheres to the `__enter__()` and `__exit__()` protocol (PEP 343). It guarantees deterministic closure of the underlying file descriptor, even if exceptions are raised during file operations.

This represents the industry-standard idiom for file handling in Python data pipelines.

In [2]:
# Utilize Python's context manager ('with' statement) for deterministic resource management
# The context manager automatically invokes __enter__() and guarantees __exit__() execution,
# ensuring the file descriptor is released even if an unhandled exception occurs
with open("./data/data.txt", "r") as f:
    # Read the full text buffer from the active file stream
    contenu = f.read()
    # Output the retrieved content to stdout
    print(contenu)

NLP.
Data analysis.
New line added.



## 3. File Writing Operations: Overwrite (`'w'`) vs. Append (`'a'`)

Python provides distinct access modes for file persistence:
- **Write Mode (`'w'`):** Truncates existing files to 0 bytes upon opening (destructive operation) or creates a new file if it does not exist.
- **Append Mode (`'a'`):** Positions the stream cursor strictly at End-Of-File (EOF), preserving all pre-existing records and writing incoming data non-destructively.

In [3]:
# Open target file in write mode ('w')
# WARNING: Write mode truncates the file to 0 bytes if it exists, or creates a new file if nonexistent
with open("./data/data.txt", "w") as f:
    # Write the first line to the stream buffer with an explicit newline delimiter ('\n')
    f.write("NLP.\n")
    # Write the second line to the stream buffer
    f.write("Data analysis.\n")

In [4]:
# Open the file in append mode ('a')
# Append mode positions the file pointer at EOF (End-Of-File), preserving pre-existing records
with open("./data/data.txt", "a") as f:
    # Append a new text record to the end of the file
    f.write("New line added.\n")

## 4. Memory-Efficient Stream Ingestion: Streaming vs. Batch Loading

When handling large NLP corpora (such as multi-gigabyte text datasets), loading the entire corpus into memory with `.read()` or `.readlines()` can lead to memory exhaustion (OOM).

- **Line-by-Line Streaming (`for line in f`):** Evaluates lines lazily as an iterator. Memory footprint remains constant O(1) relative to file size.
- **Batch List Loading (`.readlines()`):** Reads all lines immediately into a Python `list` in memory (O(N) space complexity).

In [5]:
# Open the target file for streaming line-by-line
with open("./data/data.txt", "r") as f:
    # Iterate directly over the file handle (buffered generator)
    # Evaluates lines lazily, yielding O(1) auxiliary memory consumption per line
    for line in f:
        # Strip trailing newline characters ('\n') and extraneous whitespace before printing
        print(line.strip())

NLP.
Data analysis.
New line added.


In [6]:
# Open file (omitting the mode defaults to read mode 'r')
with open("./data/data.txt") as f:
    # Read all remaining lines from the stream into a Python list of strings in RAM
    # Note: Preserves newline characters ('\n') at the end of each list element
    lignes = f.readlines()

# Output the list containing all file lines
print(lignes)

['NLP.\n', 'Data analysis.\n', 'New line added.\n']


## 5. File Cursor and Pointer Positioning (`seek()`)

The file stream maintains an internal byte offset cursor indicating where the next read or write operation will take place.
- After `.read()`, the cursor resides at EOF. Subsequent read calls return an empty string `""`.
- `seek(offset, whence=0)` repositions the cursor:
  - `offset`: Byte displacement.
  - `whence=0`: Relative to the start of the file.

In [7]:
# Open file descriptor to demonstrate stream pointer navigation
my_file = open("./data/data.txt", "r")

# Initial read: stream pointer advances from byte offset 0 to End-Of-File (EOF)
content = my_file.read()
print(content)
print("-------")

# Reposition the file pointer back to the beginning of the file (byte offset 0)
# seek(offset, whence=0): 0 designates the absolute byte position from stream origin
my_file.seek(0)

# Secondary read from byte 0: successfully reads all lines because the pointer was rewound
lines = my_file.readlines()
print(lines)

# Explicitly close the file descriptor to avoid descriptor leaks
my_file.close()

NLP.
Data analysis.
New line added.

-------
['NLP.\n', 'Data analysis.\n', 'New line added.\n']


## 6. Character Encodings & Multilingual NLP (UTF-8 and Arabic Script)

Text processing across non-Latin scripts (e.g., Arabic, Hebrew, Chinese, Cyrillic) requires explicit encoding standards.
- ASCII and Latin-1 (ISO-8859-1) represent characters using 7 or 8 bits, causing `UnicodeEncodeError` or corrupted text (*mojibake*) on non-Latin corpora.
- **UTF-8** is a variable-width Unicode encoding (1 to 4 bytes per character) that covers all international Unicode code points.

Explicitly specifying `encoding="utf-8"` is mandatory in internationalized NLP pipelines.

In [8]:
# Open file with explicit UTF-8 character encoding in write mode
# Standard ASCII / Latin-1 cannot represent non-Latin alphabets without encoding errors
with open("./data/arabic_doc.txt", "w", encoding="utf-8") as f:
    # Write Arabic Unicode text characters to the file
    f.write("مرحبا بالعالم\n")
    f.write("تحليل البيانات.\n")

# Reopen the document in read mode with explicit UTF-8 decoding
with open("./data/arabic_doc.txt", "r", encoding="utf-8") as f:
    # Read and print the decoded Unicode text to verify character preservation
    print(f.read())

مرحبا بالعالم
تحليل البيانات.



## 7. Structured Tabular Data Engineering: Ingestion & CSV Export

In NLP and machine learning, text is often stored alongside structured metadata (e.g., author, timestamp, class labels).
Pandas provides high-performance data structures (`DataFrame`) with serialization primitives:
- `DataFrame.to_csv()`: Serializes tabular structures with granular controls (`index`, `sep`, `encoding`).

In [10]:
# Import pandas for structured data manipulation and tabular serialization
import pandas as pd

# Define a dictionary containing structured tabular records
data = {
    "Name": ["Alice", "Bob", "Charlie"],
    "Age": [25, 30, 35],
    "Ville": ["Paris", "Lyon", "Marseille"]
}

# Construct a 2D labeled DataFrame from the dictionary
df = pd.DataFrame(data)

# Export DataFrame to CSV format:
# - index=False: Exclude default integer index column from export
# - sep=",": Use comma as delimiter
# - encoding="utf-8": Enforce UTF-8 encoding for special and accented characters
df.to_csv("./data/donnees.csv", index=False, sep=",", encoding="utf-8")

## 8. Tabular Ingestion with Parameter Customization & Multi-Format Serialization

Datasets frequently contain metadata headers, inconsistent delimiters, or proprietary binary formats (such as Excel `.xlsx`).
- `skiprows=N`: Skips the first N rows to bypass header comments or corrupted metadata.
- `to_excel()`: Persists datasets into multi-sheet Openpyxl workbooks with explicit sheet naming.

In [ ]:
# Import pandas and numpy for numerical and tabular processing
import pandas as pd
import numpy as np

# Ingest CSV file skipping the initial row (e.g., when skipping corrupted header/metadata)
df_csv = pd.read_csv("./data/donnees.csv", skiprows=1)

# Ingestion from Excel format (reference demonstration using openpyxl engine)
# df_excel = pd.read_excel("./data/donnees.xlsx")

# Display the top records of the ingested DataFrame
print(df_csv.head())
# print(df_excel.head())

# Generate a synthetic matrix of shape (100, 2) drawn from a uniform distribution [0, 1)
data = pd.DataFrame(np.random.rand(100, 2), columns=['Score1', 'Score2'])

# Serialize synthetic dataset to CSV without index column
data.to_csv("./data/output_scores.csv", index=False)

# Serialize synthetic dataset to Microsoft Excel format (.xlsx) with designated sheet name
data.to_excel("./data/output_excel.xlsx", sheet_name="Feuil1", index=False)

     Alice  25      Paris
0      Bob  30       Lyon
1  Charlie  35  Marseille


## 9. Binary Stream Ingestion & PDF Document Parsing (`PyPDF2`)

Unstructured text is frequently encapsulated within compiled binary formats such as Adobe Portable Document Format (PDF).
- PDFs must be opened in binary read mode (`'rb'`).
- The `PyPDF2.PdfReader` object navigates the internal document cross-reference table, extracts page counts, and decodes textual streams via `.extract_text()`.

In [ ]:
# Import PyPDF2 library for binary PDF stream parsing and manipulation
import PyPDF2

# Open PDF file in binary read mode ('rb') - required for non-text binary streams
file = open("./data/sample.pdf", "rb")

# Initialize PdfReader object to parse the PDF document structure
pdf_reader = PyPDF2.PdfReader(file)

# Extract and display the total page count from the document metadata
print("Nombre de pages:", len(pdf_reader.pages))

# Retrieve the first page object (0-indexed)
page_one = pdf_reader.pages[0]

# Extract raw text stream from the first page
text_page_1 = page_one.extract_text()

# Display the first 200 characters of the extracted text
print(text_page_1[:200])

# Initialize an aggregation list to collect text from all document pages
full_text = []

# Iterate across all pages in the PDF document sequentially
for p in range(len(pdf_reader.pages)):
    page = pdf_reader.pages[p]
    # Extract text from current page and append to corpus accumulator
    full_text.append(page.extract_text())

# Close the binary file handle to release operating system resources
file.close()

# Print the complete extracted multi-page corpus
print(full_text)

Nombre de pages: 9
RÉSUMÉ DE COURS
Machine Learning — Introduction, Catégories & Apprentissage par Renforcement
 Pr. Boutaina Ettetuani | Université Mundiapolis | 2025-2026
 
 CHAPITRE 1
 
 Introduction au Machine Learn
["RÉSUMÉ DE COURS\nMachine Learning — Introduction, Catégories & Apprentissage par Renforcement\n Pr. Boutaina Ettetuani | Université Mundiapolis | 2025-2026\n \n CHAPITRE 1\n \n Introduction au Machine Learning\n \n1. Définition\nLe Machine Learning (ML) est une branche de l'IA qui permet à une machine d'apprendre automatiquement à partir\nde données, sans règles programmées explicitement, pour faire des prédictions ou aider à la décision.\nFormule clé : y = f(x) + bruit\n2. Hiérarchie des Concepts\nData Science ⊃ IA ⊃ ML ⊃ Deep Learning. Le Data Mining est complémentaire : il extrait des patterns cachés dans\nde grandes bases de données. Le ML se distingue de la programmation classique : au lieu de coder des règles, on\nfournit des données et des résultats, et la mach

## 10. PDF Document Assembly & Stream Concatenation

Advanced corpus preparation may require restructuring multi-page publications:
- `PdfWriter`: Creates new PDF documents by adding selected page objects and serializing them in binary write mode (`'wb'`).
- `PdfMerger`: Sequentially concatenates multiple independent PDF streams into a unified document.

In [ ]:
# Import specialized PyPDF2 utility classes for document composition and merging
from PyPDF2 import PdfMerger, PdfWriter, PdfReader

# --- Single Page Extraction and Creation ---
# Open the source PDF document in binary read mode
f = open("./data/sample.pdf", "rb")
reader = PdfReader(f)

# Instantiate a PdfWriter object to generate a new PDF document
writer = PdfWriter()

# Extract the first page and append it to the writer buffer
first_page = reader.pages[0]
writer.add_page(first_page)

# Open destination binary stream and serialize the single-page document
output_file = open("./data/sample_copy.pdf", "wb")
writer.write(output_file)

# Close open streams
output_file.close()
f.close()

# --- Multi-Document Merging ---
# Instantiate a PdfMerger object to concatenate multiple PDF streams
merger = PdfMerger()

# Append the original PDF and the single-page copy to the merger queue
merger.append(PdfReader(open("./data/sample.pdf", "rb")))
merger.append(PdfReader(open("./data/sample_copy.pdf", "rb")))

# Write the concatenated binary document to disk
merger.write("./data/merged.pdf")

# Close merger to free underlying file handles
merger.close()

## 11. Lexical Pattern Matching with Regular Expressions (`re`)

Regular expressions provide deterministic lexical analysis and token identification in NLP pre-processing.
- `re.search(pattern, text)`: Scans the target string for the first match, returning a `Match` object containing match boundaries (`.span()`) and content (`.group()`).
- `re.findall(pattern, text)`: Exhaustively returns all non-overlapping pattern occurrences as a list of strings.

In [ ]:
# Import regular expressions module (re) for lexical analysis and pattern matching
import re

# Target corpus sentence containing repetitive lexical entities
texte = "Le cours de Python commence a 14h00 dans la salle 204Python."

# Search for the first occurrence of the pattern 'Python' in the text stream
match = re.search(r'Python', texte)

# Check if a Match object was found
if match:
    print("Mot trouvé:")
    # Extract the matched substring using .group()
    print("Texte extrait:", match.group())
    # Retrieve the starting and ending index offsets using .span()
    print("Position:", match.span())
else:
    print("Mot non trouvé.")

Mot trouvé:
Texte extrait: Python
Position: (12, 18)


In [ ]:
# Import regular expressions module (re)
import re

# Sample target text string
texte = "Le cours de Python commence a 14h00 dans la salle 204Python."

# re.findall() returns all non-overlapping matches of pattern in string as a list of strings
l = re.findall(r'Python', texte)

# Display the extracted match list
print(l)

['Python', 'Python']


## 12. Summary & Engineering Principles

### Key Takeaways for NLP Pipelines
1. **Resource Lifecycle:** Always prefer context managers (`with`) to safeguard against file descriptor exhaustion.
2. **Memory Efficiency:** For multi-gigabyte corpora, stream records line-by-line using generator iteration rather than caching full lists via `.readlines()`.
3. **Encoding Determinism:** System default encodings vary across OS distributions (UTF-8 on Linux, CP-1252/CP-1256 on Windows). Explicitly declare `encoding="utf-8"` to prevent corpus corruption.
4. **Multi-Format Ingestion:** Standardize structured datasets via Pandas CSV/Excel exporters, and leverage binary decoders (`PyPDF2`) for document-level corpus extraction.
5. **Lexical Validation:** Utilize regular expression spans and groupings to validate tokens and extract domain-specific terminology prior to tokenization.